In [1]:
import os, sys
# optional: make notebook treat repo root as working dir
os.chdir('..')
sys.path.insert(0, os.path.abspath('backend'))

In [2]:
# Jupyter magic to display all columns of a dataframe
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)  # Show all columns in the dataframe
pd.set_option('display.max_rows', 10)     # Show all rows in the dataframe (

In [138]:
import json

train_questions_path = "./backend/evaluation/data/dblp-quad/train.questions.json"
train_answers = "./backend/evaluation/data/dblp-quad/train.answers.json"

with open(train_questions_path) as f:
    raw_data = json.load(f)
    train_questions = pd.DataFrame.from_records(raw_data["questions"])

with open(train_answers) as f:
    raw_data = json.load(f)
    train_answers = pd.DataFrame.from_records(raw_data["answers"])
    new_df = pd.json_normalize(train_answers.answer)
    train_answers.drop(columns=['answer'], inplace=True)
    train_answers = train_answers.join(new_df)


In [139]:
# train_questions = train_questions[train_questions["query"].apply(lambda q: q['sparql'].startswith("SELECT"))] # Filter query type for SELECT
train_questions.reset_index(drop=True, inplace=True)
print("Total questions:", len(train_questions))

# Filter answers to only include those that correspond to the filtered questions
train_answers = train_answers[train_answers["id"].isin(train_questions["id"])]
train_answers.reset_index(drop=True, inplace=True)
print("Total answers:", len(train_answers))

train_questions.head(3)

Total questions: 7000
Total answers: 7000


,id,query_type,question,paraphrased_question,query,template_id,entities,relations,temporal,held_out
0,Q0001,SINGLE_FACT,{'string': 'What are the papers written by the...,{'string': 'Which papers did the author Wazir ...,{'sparql': 'SELECT DISTINCT ?answer WHERE { ?a...,TC01,[<https://dblp.org/pid/211/3355>],[<https://dblp.org/rdf/schema#authoredBy>],False,False
1,Q0002,SINGLE_FACT,{'string': 'What is the Wikidata ID of Yvo Des...,{'string': 'The author Y. Desmedt is associate...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC04,[<https://dblp.org/pid/d/YvoDesmedt>],[<https://dblp.org/rdf/schema#wikidata>],False,False
2,Q0003,SINGLE_FACT,{'string': 'What is the primary affiliation of...,{'string': 'Leandro Krug Wives is primarily af...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC02,[<https://dblp.org/pid/w/LeandroKrugWives>],[<https://dblp.org/rdf/schema#primaryAffiliati...,False,False


In [140]:
index = 2000
train_questions.loc[index].to_dict()

{'id': 'Q2001',
 'query_type': 'DOUBLE_NEGATION',
 'question': {'string': 'Was a paper not not co-authored by Larry S. Jackson and Huy-Vo Van?'},
 'paraphrased_question': {'string': "Didn't Jackson, L. S. and Huy-Vo Van not co-author a paper?"},
 'query': {'sparql': 'ASK { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/03/326> . ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/25/10708> }'},
 'template_id': 'TC53',
 'entities': ['<https://dblp.org/pid/03/326>',
  '<https://dblp.org/pid/25/10708>'],
 'relations': ['<https://dblp.org/rdf/schema#authoredBy>'],
 'temporal': False,
 'held_out': False}

In [141]:
train_answers.loc[index].to_dict()

{'id': 'Q2001',
 'head.link': [],
 'head.vars': nan,
 'results.distinct': nan,
 'results.ordered': nan,
 'results.bindings': nan,
 'boolean': False}

In [142]:
# print all the unique relations in the dataset
unique_relations = set(rel for rels in train_questions["relations"] for rel in rels)
unique_relations

{'<https://dblp.org/rdf/schema#authoredBy>',
 '<https://dblp.org/rdf/schema#numberOfCreators>',
 '<https://dblp.org/rdf/schema#primaryAffiliation>',
 '<https://dblp.org/rdf/schema#publishedIn>',
 '<https://dblp.org/rdf/schema#title>',
 '<https://dblp.org/rdf/schema#webpage>',
 '<https://dblp.org/rdf/schema#wikidata>',
 '<https://dblp.org/rdf/schema#yearOfPublication>'}

In [143]:
def filter_relation(questions_df, relation):
    return questions_df[questions_df["relations"].apply(lambda rels: relation in rels)]

publishedIn_relation = "<https://dblp.org/rdf/schema#publishedIn>"
publishedIn_questions = filter_relation(train_questions, publishedIn_relation)
publishedIn_questions.count()

id                      2327
query_type              2327
question                2327
paraphrased_question    2327
query                   2327
template_id             2327
entities                2327
relations               2327
temporal                2327
held_out                2327
dtype: int64

In [144]:
#  filter_relation(train_questions,  '<https://dblp.org/rdf/schema#primaryAffiliation>').reset_index().loc[10].to_dict()

In [145]:
# List all entities that are NOT IRIs ("<...>") 
non_iri_entities = list(set(ent for ents in train_questions["entities"] for ent in ents if not ent.startswith("<")))
non_iri_entities


['Sensors',
 'ACCV (3)',
 'Intelligent Environments',
 'IEEE J. Sel. Areas Commun.',
 'ISM',
 'NEMS',
 'Artif. Intell. Law',
 'J. Glob. Optim.',
 'ECML/PKDD (2)',
 'SEKE',
 'CICC',
 'Visual Information Processing',
 'J. Exp. Theor. Artif. Intell.',
 'J. Networks',
 'ISNN (1)',
 'IROS',
 'ICIA',
 'Bioinform.',
 'J. Inf. Process. Syst.',
 'CSEDU (2)',
 'UCAmI',
 'ISPAN',
 'ACM Symposium on Document Engineering',
 'Medical Image Anal.',
 'HMD Prax. Wirtsch.',
 'IEEE Trans. Biomed. Eng.',
 'IEEE Trans. Geosci. Remote. Sens.',
 'Hum. Comput.',
 'Allerton',
 'ICME',
 'Fundam. Informaticae',
 'Intell. Autom. Soft Comput.',
 'Eur. Trans. Telecommun.',
 'ROBIO',
 'SMM4H@AMIA',
 'IJCNN',
 'IEEE Trans. Commun.',
 'ICOIN',
 'IEEE Trans. Netw. Sci. Eng.',
 'SAAIP@IJCAI',
 'SoMeT',
 'Int. J. Comput. Intell. Syst.',
 'APSIPA',
 'Australas. J Comb.',
 'Inf. Softw. Technol.',
 'CW',
 'ACC',
 'J. Softw.',
 'CAIP',
 'Multim. Tools Appl.',
 'LATIN',
 'ANSS',
 'AGILE',
 'CDC/ECC',
 'J. Documentation',
 'IS

In [147]:
from httpx import AsyncClient

DBLP_VENUE_API= "https://dblp.org/search/venue/api"


async def link_venue_to_dblp(venue_name: str, max_results: int = 20):
    params = {"q": venue_name, "format": "json", "h": max_results}

    async with AsyncClient(
        timeout=10.0,
        headers={"User-Agent": "DBLP-Quad-Linker/1.0 (+https://github.com/jeffry/dblp-quad-linker)"}
    ) as client:
        resp = await client.get(DBLP_VENUE_API, params=params)
    resp.raise_for_status()
    data = resp.json()
    hits = data.get("result", {}).get("hits", {}).get("hit", [])
    candidates = []
    for h in hits:
        info = h.get("info", {})
        # Extract the venue key from the URL (e.g., https://dblp.org/db/conf/nips/ -> conf/nips)
        url = info.get("url", "")
        key = url.replace("https://dblp.org/db/", "").rstrip("/") if url else None
        candidates.append({
            "name":    info.get("venue"),
            "url":     url,
            "dblp_id": key,
            "hit_id":  h.get("@id"),  # Also store the hit ID
            "type": "Venue"
        })
    return candidates


In [148]:
import asyncio
# Get data from DBLP for all non-IRI entities
# data = {}

In [176]:
print(len(non_iri_entities), len(data.keys()))
for entity in non_iri_entities:
    if entity in data:
        continue
    res  = await link_venue_to_dblp(entity)
    data[entity] = res
    await asyncio.sleep(3)  # Sleep to avoid hitting rate limits


101 100


In [190]:
matches = {}

for entity, candidates in data.items():
    
    entity = entity.strip()
    match = None
    in_brackets = f"({entity.strip()})"
    for candidate in candidates:
        candidate_name = candidate["name"].strip()
        # endswith (xxx)
        if candidate_name.endswith(in_brackets):
            match = candidate
            break 
        if candidate_name.lower().endswith(in_brackets.lower()):
            match = candidate
            break

        # endswith xxx
        if candidate_name.endswith(entity):
            match = candidate
            break
        if candidate_name.lower().endswith(entity.lower()):
            match = candidate
            break

    if match is None and len(candidates) == 1:
        match = candidates[0]

    if match:
        matches[entity] = {}
        matches[entity]["match"] = match
        # matches[entity]["candidates"] = candidates


print(len(matches))

67


In [191]:
# manual matching
matches["ACCV (3)"] = {"match": {"url": "https://dblp.org/db/conf/accv"}}
matches['Artif. Intell. Law'] = {"match": {'name': 'Artificial Intelligence and Law', 'url': 'https://dblp.org/db/journals/ail/', 'dblp_id': 'journals/ail', 'hit_id': '550', 'type': 'Venue'}}
matches['J. Glob. Optim.'] = {"match": {'name': 'Journal of Global Optimization', 'url': 'https://dblp.org/db/journals/jgo/', 'dblp_id': 'journals/jgo', 'hit_id': '6578', 'type': 'Venue'}}

In [192]:
missing = []

for index, entity in enumerate(non_iri_entities):

    if entity in ('ECML/PKDD (2)', 'Visual Information Processing'):
        continue

    if entity not in matches:
        og_row = train_questions[train_questions["entities"].apply(lambda ents: entity in ents)]
        question = og_row["question"].values[0]["string"]
        missing.append({
            "entity": entity,
            "question": question,
        })

        print("Question:", repr(question), og_row["id"].values[0])
        print("DBLP candidates for entity:", repr(non_iri_entities[index]))
        print(data[non_iri_entities[index]])

        break
        

Question: "Did the authors of the publication 'An Analysis of Fault Effects and Propagations in AVR Microcontroller ATmega103(L)' also publish a publication in J. Networks?" Q4653
DBLP candidates for entity: 'J. Networks'
[{'name': 'Journal of Wireless Mobile Networks, Ubiquitous Computing, and Dependable Applications (JoWUA)', 'url': 'https://dblp.org/db/journals/jowua/', 'dblp_id': 'journals/jowua', 'hit_id': '6731', 'type': 'Venue'}, {'name': 'Journal of Communications and Information Networks', 'url': 'https://dblp.org/db/journals/jcin/', 'dblp_id': 'journals/jcin', 'hit_id': '6498', 'type': 'Venue'}, {'name': 'Journal of Communications and Networks', 'url': 'https://dblp.org/db/journals/jcn/', 'dblp_id': 'journals/jcn', 'hit_id': '6499', 'type': 'Venue'}, {'name': 'Journal of Computer Networks and Communications', 'url': 'https://dblp.org/db/journals/jcnc/', 'dblp_id': 'journals/jcnc', 'hit_id': '6518', 'type': 'Venue'}, {'name': 'Journal of High Speed Networks', 'url': 'https://d

In [198]:
# create new column in train_questions called "processed_entities" which contains the IRIs and matches entities
def get_processed_entities(entities):
    res = []
    for entity in entities:
        if entity.startswith("<"):
            res.append(entity)
        elif entity in matches:
            match = matches[entity].get("match")
            if match and "url" in match:
                # Wrap the URL in angle brackets to form an IRI
                res.append(f"<{match['url']}>")
    return res
train_questions["processed_entities"] = train_questions["entities"].apply(get_processed_entities)


# Save file to disk
with open("./backend/evaluation/data/dblp-quad/train_questions_processed.json", "w") as f:
    json.dump(train_questions.to_dict(orient="records"), f, indent=2)
